In [2]:
import pandas as pd
import os
from pathlib import Path
import csv

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [2]:
dataset_specs = {
    "obis_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OBIS\obis_final.parquet"),
    },
    "gbif_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet"),
    },
    "ogsl_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OGSL\ogsl_final.parquet"),
    },
    "cb_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\CB\cb_final.parquet"),
    },
}

def normalize_type(value):
    t = str(value).strip().lower()
    # integers (including pandas nullable Int64 etc.)
    if any(x in t for x in ["int", "integer", "smallint", "bigint"]) or any(x in t for x in ["int64", "int32", "int16"]):
        return "integer"
    # floats (including pandas nullable Float64)
    if any(x in t for x in ["float", "double", "decimal", "numeric", "real"]) or any(x in t for x in ["float64", "float32"]):
        return "float"
    # datetimes
    if any(x in t for x in ["datetime", "timestamp", "date", "time"]) or "datetime64" in t:
        return "datetime"
    # booleans
    if "bool" in t or t == "boolean":
        return "boolean"
    # categorical types
    if any(x in t for x in ["category", "categorical"]):
        return "category"
    # strings: treat object, str, pandas string, pyarrow string, etc. as the same
    if any(x in t for x in ["string", "object", "char", "text", "varchar", "str", "pyarrow.string", "string[python]", "string[pyarrow]"]):
        return "string"
    # binary/bytes
    if "bytes" in t or "binary" in t:
        return "binary"
    return t

# Load all parquets
parquets = {}
schema_rows = []
comparison_rows = []

for dataset, spec in dataset_specs.items():
    parquet_path = spec["parquet"]
    if not parquet_path.exists():
        schema_rows.append({"dataset": dataset, "status": "missing parquet"})
        continue
    
    df = pd.read_parquet(parquet_path)
    parquets[dataset] = df
    schema_rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "status": "loaded"
    })

# Compare parquets to each other
datasets = list(parquets.keys())
all_columns = set()
for df in parquets.values():
    all_columns.update(df.columns)

for col in sorted(all_columns):
    for dataset1, dataset2 in [(datasets[i], datasets[j]) for i in range(len(datasets)) for j in range(i+1, len(datasets))]:
        df1 = parquets[dataset1]
        df2 = parquets[dataset2]
        
        in_df1 = col in df1.columns
        in_df2 = col in df2.columns
        
        if in_df1 and in_df2:
            type1 = normalize_type(str(df1[col].dtype))
            type2 = normalize_type(str(df2[col].dtype))
            if type1 != type2:
                comparison_rows.append({
                    "column": col,
                    "dataset1": dataset1,
                    "type1": str(df1[col].dtype),
                    "dataset2": dataset2,
                    "type2": str(df2[col].dtype),
                    "issue": "dtype mismatch"
                })
        elif in_df1 and not in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset1} only"
            })
        elif not in_df1 and in_df2:
            comparison_rows.append({
                "column": col,
                "dataset1": dataset1,
                "dataset2": dataset2,
                "issue": f"present in {dataset2} only"
            })

schema_report = pd.DataFrame(schema_rows)
comparison_report = pd.DataFrame(comparison_rows)

# Save to CSV
comparison_report.to_csv("parquet_comparison_report.csv", index=False)
schema_report.to_csv("parquet_schema_report.csv", index=False)

print(f"Saved comparison_report to: parquet_comparison_report.csv")
print(f"Saved schema_report to: parquet_schema_report.csv")

# Harmonise dtypes across dataframes and merge the four datasets
print("Harmonizing dtypes and merging datasets...")
# Determine target type for each column based on present types
target_types = {}
for col in sorted(all_columns):
    types = []
    for name, df in parquets.items():
        if col in df.columns:
            types.append(normalize_type(str(df[col].dtype)))
    if 'string' in types:
        target = 'string'
    elif 'datetime' in types:
        target = 'datetime'
    elif 'float' in types:
        target = 'float'
    elif 'integer' in types:
        target = 'integer'
    elif 'boolean' in types:
        target = 'boolean'
    elif 'category' in types:
        target = 'category'
    else:
        target = None
    target_types[col] = target

# Cast columns in each dataframe to the chosen target type (add missing columns)
for name, df in parquets.items():
    for col, target in target_types.items():
        if col not in df.columns:
            df[col] = pd.Series([pd.NA] * len(df), index=df.index)
        if target == 'string':
            df[col] = df[col].astype('string')
        elif target == 'datetime':
            df[col] = pd.to_datetime(df[col], errors='coerce')
        elif target == 'float':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('float64')
        elif target == 'integer':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
        elif target == 'boolean':
            try:
                df[col] = df[col].astype('boolean')
            except Exception:
                df[col] = df[col].map({True: True, False: False}).astype('boolean')
        elif target == 'category':
            df[col] = df[col].astype('category')

# Concatenate in a consistent column order
col_order = sorted(all_columns)
merged = pd.concat([parquets[d] for d in datasets], ignore_index=True, sort=False)[col_order]

# Save merged dataset
merged_path = Path('All_data/merged_all.parquet')
merged.to_parquet(merged_path, index=False)
print(f"Saved merged to: {merged_path}")

schema_report, comparison_report, merged

Saved comparison_report to: parquet_comparison_report.csv
Saved schema_report to: parquet_schema_report.csv
Harmonizing dtypes and merging datasets...
Saved merged to: All_data\merged_all.parquet


(      dataset   n_rows  n_columns  status
 0  obis_final  1533558         62  loaded
 1  gbif_final  2588475         62  loaded
 2  ogsl_final     7257         62  loaded
 3    cb_final   243609         62  loaded,
               column    dataset1    type1    dataset2    type2           issue
 0                 ID  obis_final    Int32  gbif_final  float64  dtype mismatch
 1                 ID  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 2                 ID  gbif_final  float64    cb_final    Int32  dtype mismatch
 3            aphiaID  obis_final    Int32  gbif_final  float64  dtype mismatch
 4            aphiaID  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 5            aphiaID  gbif_final  float64    cb_final    Int32  dtype mismatch
 6       endDayOfYear  obis_final    Int32  gbif_final  float64  dtype mismatch
 7       endDayOfYear  gbif_final  float64  ogsl_final    Int32  dtype mismatch
 8       endDayOfYear  gbif_final  float64    cb_final    Int32 

In [3]:
# Calculate coverage (non-null percentage) for each column
coverage_data = []
for col in col_order:
    total_rows = len(merged)
    non_null_count = merged[col].notna().sum()
    coverage_pct = (non_null_count / total_rows) * 100
    coverage_data.append({
        "column": col,
        "non_null_count": non_null_count,
        "total_rows": total_rows,
        "coverage_percent": coverage_pct
    })

coverage_report = pd.DataFrame(coverage_data)
coverage_report.to_csv("column_coverage.csv", index=False)
print(f"Saved column coverage report to: column_coverage.csv")
print(coverage_report)

Saved column coverage report to: column_coverage.csv
                    column  non_null_count  total_rows  coverage_percent
0                       ID               0     4372899          0.000000
1   acceptedScientificName         4372899     4372899        100.000000
2         acceptedTaxonKey         2588475     4372899         59.193569
3                  aphiaID         4336707     4372899         99.172357
4              areaSampled          223537     4372899          5.111872
..                     ...             ...         ...               ...
57            verbatimDate         4249281     4372899         97.173088
58              verbatimID         4045497     4372899         92.512930
59  verbatimScientificName         4364974     4372899         99.818770
60               waterBody         3673932     4372899         84.015935
61                    year         4172079     4372899         95.407623

[62 rows x 4 columns]


# Deduplicate

Between-database deduplication

In [4]:
# Create a source priority mapping
source_priority = {
    'OGSL': 1,
    'CRITTERBASE': 2,
    'OBIS': 3,
    'GBIF': 4
}

# Deduplication key columns
key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']

# --- Preprocess / normalize fields so "missing" is consistent ---
# Work on a copy and attach a stable row id for reliable keep/remove tracing
merged = merged.copy()
if '_row_id' not in merged.columns:
    merged['_row_id'] = merged.index.astype('int64')

# Normalize sourceDatabase (strip/case)
merged['sourceDatabase_norm'] = (
    merged['sourceDatabase'].astype('string')
          .str.strip()
          .str.upper()
)
merged['sourceDatabase_norm'] = merged['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Normalize numeric key columns (coerce invalid values to NaN)
for num_col in ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']:
    if num_col in merged.columns:
        merged[num_col] = pd.to_numeric(merged[num_col], errors='coerce')

# Identify rows with all key deduplication columns non-null (these are eligible for dedup)
mask = merged[key_cols].notna().all(axis=1)

# Split into rows to deduplicate and rows to keep as-is
to_dedup = merged[mask].copy()
to_keep = merged[~mask].copy()

# For each place+date combination, count how many distinct databases have it
place_date_db_count = to_dedup.groupby(key_cols)['sourceDatabase_norm'].nunique().reset_index()
place_date_db_count.columns = key_cols + ['db_count']

# Merge this count back into to_dedup
to_dedup = to_dedup.merge(place_date_db_count, on=key_cols, how='left')

# Only rows where place+date appears in MULTIPLE databases are eligible for removal
cross_db_duplicates = to_dedup[to_dedup['db_count'] > 1].copy()
within_db_only = to_dedup[to_dedup['db_count'] == 1].copy()

if len(cross_db_duplicates) > 0:
    # For cross-database duplicates, keep all rows from the highest-priority database per place+date
    cross_db_duplicates['_priority'] = (
        cross_db_duplicates['sourceDatabase_norm'].map(source_priority).fillna(999)
    )
    # Keep sorted view for debugging readability
    cross_db_duplicates_sorted = cross_db_duplicates.sort_values(by=key_cols + ['_priority'], na_position='last')

    # Compute best priority within each place+date group and keep all rows at that priority
    best_priority = cross_db_duplicates.groupby(key_cols)['_priority'].transform('min')
    cross_db_duplicates_deduped = cross_db_duplicates[cross_db_duplicates['_priority'] == best_priority].copy()
else:
    cross_db_duplicates_deduped = cross_db_duplicates.copy()

# Within-database duplicates (same place/date in only one database) are kept as-is
within_db_only = within_db_only.copy()

# Compute which rows were removed during cross-database deduplication using stable row ids
if len(cross_db_duplicates) > 0:
    kept_row_ids = set(cross_db_duplicates_deduped['_row_id'].tolist())
    removed = cross_db_duplicates[~cross_db_duplicates['_row_id'].isin(kept_row_ids)].copy()
else:
    removed = cross_db_duplicates.copy()

# Drop helper columns before final concat/save
drop_cols = [c for c in ['_priority', 'db_count'] if c in cross_db_duplicates_deduped.columns]
cross_db_duplicates_deduped = cross_db_duplicates_deduped.drop(columns=drop_cols)
within_db_only = within_db_only.drop(columns=['db_count'] if 'db_count' in within_db_only.columns else [])

# Combine deduplicated cross-database records with within-database records
to_dedup_deduped = pd.concat([cross_db_duplicates_deduped, within_db_only], ignore_index=True)

# Concatenate back together
merged_deduped = pd.concat([to_dedup_deduped, to_keep], ignore_index=True)

# Do not persist helper row id
if '_row_id' in merged_deduped.columns:
    merged_deduped = merged_deduped.drop(columns=['_row_id'])

print(f"Original merged dataset: {len(merged)} rows")
print(f"After deduplication: {len(merged_deduped)} rows")
print(f"Rows deduplicated: {len(merged) - len(merged_deduped)}")
print(f"Cross-database duplicates found: {len(cross_db_duplicates)}")
print(f"Within-database records preserved: {len(within_db_only)}")

# Save the deduplicated dataset
merged_deduped.to_parquet('All_data/merged_all_deduplicated.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet")

if len(cross_db_duplicates) > 0:
    removed_counts = removed['sourceDatabase_norm'].value_counts(dropna=False).rename_axis('sourceDatabase').reset_index(name='removed_count')
    print('\nRemoved counts by sourceDatabase:')
    print(removed_counts)
    removed_counts.to_csv('dedup_removed_by_sourceDatabase.csv', index=False)
    print('Saved removed counts to: dedup_removed_by_sourceDatabase.csv')
    
    print('\nExamples of removed rows (head):')
    print(removed.head(10))
    
    removed.head(50).to_parquet('All_data/dedup_removed_sample.parquet', index=False)
    print('Saved sample removed rows to: All_data/dedup_removed_sample.parquet')
else:
    print("\nNo cross-database duplicates found.")

Original merged dataset: 4372899 rows
After deduplication: 4070651 rows
Rows deduplicated: 302248
Cross-database duplicates found: 614689
Within-database records preserved: 2708384
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

Removed counts by sourceDatabase:
  sourceDatabase  removed_count
0           GBIF         292101
1           OBIS          10147
Saved removed counts to: dedup_removed_by_sourceDatabase.csv

Examples of removed rows (head):
     ID      acceptedScientificName  acceptedTaxonKey   aphiaID  areaSampled  \
119 NaN  Archypolycope semipunctata              <NA>  423158.0          NaN   
350 NaN                      Eualus              <NA>  106986.0          NaN   
439 NaN      Sclerochilus contortus              <NA>  127951.0          NaN   
441 NaN       Cytheropteron hamatum              <NA>  128017.0          NaN   
460 NaN    Microcythere medistriata              <NA>  463384.0          NaN   
690 NaN         Sclerocrangon ferox      

Within-database deduplication

In [5]:
# Remove within-database duplicates using aphiaID
# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.

# Load the deduplicated dataset
merged_deduped = pd.read_parquet('All_data/merged_all_deduplicated.parquet').copy()
if '_row_id' not in merged_deduped.columns:
    merged_deduped['_row_id'] = merged_deduped.index.astype('int64')

# Normalize sourceDatabase if not already done
if 'sourceDatabase_norm' not in merged_deduped.columns:
    merged_deduped['sourceDatabase_norm'] = (
        merged_deduped['sourceDatabase'].astype('string')
              .str.strip()
              .str.upper()
    )
    merged_deduped['sourceDatabase_norm'] = merged_deduped['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Define within-database deduplication columns
within_db_cols = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']

# Only rows with all relevant columns populated are eligible for within-database dedup
required_numeric_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']
required_text_cols = ['sourceDatabase_norm', 'aphiaID']

numeric_complete = merged_deduped[required_numeric_cols].notna().all(axis=1)
text_complete = merged_deduped[required_text_cols].apply(
    lambda s: s.astype('string').str.strip().notna() & (s.astype('string').str.strip() != '')
).all(axis=1)
within_db_mask = numeric_complete & text_complete

# Split into rows eligible for within-database dedup and rows to keep as-is
to_deduplicate_within = merged_deduped[within_db_mask].copy()
to_keep_within = merged_deduped[~within_db_mask].copy()

# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.
preference_cols = ['individualCount', 'biomass', 'gear', 'areaSampled']
preference_present_cols = [col for col in preference_cols if col in to_deduplicate_within.columns]
preference_scores = pd.Series(0, index=to_deduplicate_within.index, dtype='int64')
for col in preference_present_cols:
    present = (
        to_deduplicate_within[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .notna()
        .astype('int64')
    )
    preference_scores = preference_scores.add(present, fill_value=0).astype('int64')

to_deduplicate_within['_preference_score'] = preference_scores

# Remove within-database duplicates (same sourceDatabase, place, date, and aphiaID)
# Keep the row with the most preferred observation fields; ties fall back to first occurrence after stable sorting
sort_cols = ['_preference_score', '_row_id']
sort_ascending = [True] * len(within_db_cols) + [False, True]
to_deduplicate_within = to_deduplicate_within.sort_values(
    by=within_db_cols + sort_cols,
    ascending=sort_ascending,
    kind='mergesort'
)
to_deduplicate_within_deduped = to_deduplicate_within.drop_duplicates(subset=within_db_cols, keep='first')

# Report what was removed via stable row id
kept_row_ids = set(to_deduplicate_within_deduped['_row_id'].tolist())
removed_within = to_deduplicate_within[~to_deduplicate_within['_row_id'].isin(kept_row_ids)].copy()

# Concatenate back together
to_deduplicate_within_deduped = to_deduplicate_within_deduped.drop(columns=['_preference_score'])
merged_within_deduped = pd.concat([to_deduplicate_within_deduped, to_keep_within], ignore_index=True)
if '_row_id' in merged_within_deduped.columns:
    merged_within_deduped = merged_within_deduped.drop(columns=['_row_id'])

print(f"Before within-database deduplication: {len(merged_deduped)} rows")
print(f"Eligible rows (all relevant columns present): {len(to_deduplicate_within)}")
print(f"After within-database deduplication: {len(merged_within_deduped)} rows")
print(f"Rows removed: {len(merged_deduped) - len(merged_within_deduped)}")

# Save the result
merged_within_deduped.to_parquet('All_data/merged_all_deduplicated.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet")

if len(removed_within) > 0:
    print(f"\nWithin-database duplicates removed by sourceDatabase:")
    print(removed_within['sourceDatabase_norm'].value_counts(dropna=False))
    print(f"\nExample removed rows:")
    print(removed_within.head(10)[['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']])
else:
    print("\nNo within-database duplicates found.")

Before within-database deduplication: 4070651 rows
Eligible rows (all relevant columns present): 2997059
After within-database deduplication: 3190400 rows
Rows removed: 880251
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

Within-database duplicates removed by sourceDatabase:
sourceDatabase_norm
OBIS           615312
GBIF           208307
CRITTERBASE     56629
OGSL                3
Name: count, dtype: int64[pyarrow]

Example removed rows:
        sourceDatabase_norm    year  startDayOfYear  decimalLongitudeStart  \
3013431         CRITTERBASE  1899.0           215.0                  20.75   
3013512         CRITTERBASE  1899.0           230.0                  41.37   
3013429         CRITTERBASE  1899.0           232.0                  22.22   
3013430         CRITTERBASE  1899.0           245.0                  15.58   
2850223         CRITTERBASE  1899.0           363.0                  15.08   
2874097         CRITTERBASE  1899.0           363.0            

In [6]:
# Export OGSL and CRITTERBASE twins (removed + kept) to Excel with status labels
if 'removed_within' not in globals() or 'to_deduplicate_within_deduped' not in globals():
    print('Run the within-database deduplication cell first to create removed_within and to_deduplicate_within_deduped.')
else:
    key_cols_verify = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']
    sources_to_export = ['OGSL', 'CRITTERBASE']

    all_twins = []

    for src in sources_to_export:
        rem_src = removed_within[removed_within['sourceDatabase_norm'] == src].copy()
        kept_src = to_deduplicate_within_deduped[to_deduplicate_within_deduped['sourceDatabase_norm'] == src].copy()

        if 'sourceID' not in rem_src.columns:
            rem_src['sourceID'] = pd.NA
        if 'sourceID' not in kept_src.columns:
            kept_src['sourceID'] = pd.NA

        export_cols = ['sourceID'] + key_cols_verify
        rem_src = rem_src[export_cols].copy()
        kept_src = kept_src[export_cols].copy()

        # Keep one representative kept row per key for matching
        kept_src_one = kept_src.drop_duplicates(subset=key_cols_verify, keep='first').copy()
        kept_src_one['status'] = 'Saved'

        rem_src['status'] = 'Removed'

        # Build pair IDs from exact key tuple so removed and kept twins are linkable
        rem_src['_pair_key'] = rem_src[key_cols_verify].astype('string').agg('||'.join, axis=1)
        kept_src_one['_pair_key'] = kept_src_one[key_cols_verify].astype('string').agg('||'.join, axis=1)

        matched_keys = set(rem_src['_pair_key']).intersection(set(kept_src_one['_pair_key']))

        rem_matched = rem_src[rem_src['_pair_key'].isin(matched_keys)].copy()
        kept_matched = kept_src_one[kept_src_one['_pair_key'].isin(matched_keys)].copy()

        rem_matched['source_group'] = src
        kept_matched['source_group'] = src

        rem_matched['pair_id'] = src + '_' + rem_matched['_pair_key'].astype('category').cat.codes.astype(str)
        key_to_pair = rem_matched.drop_duplicates('_pair_key').set_index('_pair_key')['pair_id'].to_dict()
        kept_matched['pair_id'] = kept_matched['_pair_key'].map(key_to_pair)

        out_cols = ['source_group', 'pair_id', 'status', 'sourceID'] + key_cols_verify
        all_twins.append(rem_matched[out_cols])
        all_twins.append(kept_matched[out_cols])

    twins_export = pd.concat(all_twins, ignore_index=True) if len(all_twins) > 0 else pd.DataFrame()
    twins_export = twins_export.sort_values(['source_group', 'pair_id', 'status']).reset_index(drop=True)

    output_path = Path('All_data/within_db_twins_ogsl_critterbase.xlsx')
    with pd.ExcelWriter(output_path) as writer:
        twins_export.to_excel(writer, sheet_name='twins', index=False)
        summary = twins_export.groupby(['source_group', 'status']).size().rename('rows').reset_index()
        summary.to_excel(writer, sheet_name='summary', index=False)

    print(f"Saved twin export to: {output_path}")
    print('Counts by source/status:')
    print(twins_export.groupby(['source_group', 'status']).size())

Saved twin export to: All_data\within_db_twins_ogsl_critterbase.xlsx
Counts by source/status:
source_group  status 
CRITTERBASE   Removed    56629
              Saved      30356
OGSL          Removed        3
              Saved          3
dtype: int64


# Filter out irrelevant datasets

Remove datasets whose records were removed in the deduplication process

In [7]:
gbif_obis_path = Path("GBIF/datasets_GBIF_OBIS.parquet")
merged_path = Path("All_data/merged_all_deduplicated.parquet")
output_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")

gbif_obis = pd.read_parquet(gbif_obis_path)

if "merged_deduplicated" in globals():
    merged_source = merged_deduplicated
else:
    merged_source = pd.read_parquet(merged_path)

def _norm_series(s):
    return (
        s.astype("string")
         .str.strip()
         .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "NA": pd.NA, "N/A": pd.NA})
    )

valid_dataset_keys = set(_norm_series(merged_source["datasetKey"]).dropna().unique())
alt_ids = _norm_series(gbif_obis["alternateIdentifier"])

keep_mask = alt_ids.isin(valid_dataset_keys)
filtered = gbif_obis.loc[keep_mask].copy()

removed_count = len(gbif_obis) - len(filtered)
print(f"Original rows: {len(gbif_obis)}")
print(f"Removed rows: {removed_count}")
print(f"Remaining rows: {len(filtered)}")

filtered.to_parquet(output_path, index=False)
print(f"Saved filtered file to: {output_path}")

Original rows: 617
Removed rows: 16
Remaining rows: 601
Saved filtered file to: GBIF\datasets_GBIF_OBIS_filtered.parquet


In [ ]:
manual_citations_path = Path("OBIS/unique_bibliographic_citations_manual.csv")
filtered_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")

# Load the manual citations file
manual_citations = pd.read_csv(manual_citations_path)

# Load the filtered datasets
filtered = pd.read_parquet(filtered_path)

# Normalize citation columns for matching
manual_citations['bibliographicCitation_norm'] = _norm_series(manual_citations['bibliographicCitation'])
filtered['citation_norm'] = _norm_series(filtered['citation'])

# Merge the Keep column from manual_citations
filtered = filtered.merge(
    manual_citations[['bibliographicCitation_norm', 'keep']],
    left_on='citation_norm',
    right_on='bibliographicCitation_norm',
    how='left'
)

# Merge any keep-like columns into one canonical Keep column
keep_like_cols = [col for col in ['Keep', 'Keep_x', 'Keep_y', 'keep'] if col in filtered.columns]
if keep_like_cols:
    canonical_keep = filtered[keep_like_cols[0]].copy()
    for col in keep_like_cols[1:]:
        canonical_keep = canonical_keep.combine_first(filtered[col])
    filtered['Keep'] = canonical_keep
    filtered = filtered.drop(columns=[col for col in keep_like_cols if col != 'Keep'])

# Drop the temporary normalized columns
filtered = filtered.drop(columns=['citation_norm', 'bibliographicCitation_norm'])

# Count missing Keep entries
missing_keep = filtered['Keep'].isna().sum()
print(f"Records lacking a Keep entry: {missing_keep}")

# Save back to the same location

filtered.to_parquet(filtered_path, index=False)print(f"Saved updated file to: {filtered_path}")

Records lacking a Keep entry: 599
Saved updated file to: GBIF\datasets_GBIF_OBIS_filtered.parquet


Add Keep column to records file

In [ ]:
keep_manual_path = Path("GBIF/merged_dataset_with_keep_manual.csv")
filtered_path = Path("GBIF/datasets_GBIF_OBIS_filtered.parquet")

# Load the keep manual file
keep_manual = pd.read_csv(keep_manual_path)

# Normalize datasetKey in keep_manual for matching
keep_manual['datasetKey_norm'] = _norm_series(keep_manual['datasetKey'])

# Normalize datasetKey in filtered for matching
filtered['datasetKey_norm'] = _norm_series(filtered['alternateIdentifier'])

# Merge the Keep column
filtered = filtered.merge(
    keep_manual[['datasetKey_norm', 'Keep']],
    on='datasetKey_norm',
    how='left'
)

# Merge any keep-like columns into one canonical Keep column
keep_like_cols = [col for col in ['Keep', 'Keep_x', 'Keep_y', 'keep'] if col in filtered.columns]
if keep_like_cols:
    canonical_keep = filtered[keep_like_cols[0]].copy()
    for col in keep_like_cols[1:]:
        canonical_keep = canonical_keep.combine_first(filtered[col])
    filtered['Keep'] = canonical_keep
    filtered = filtered.drop(columns=[col for col in keep_like_cols if col != 'Keep'])

# Count missing Keep entries
missing_keep = filtered['Keep'].isna().sum()
print(f"Datasets lacking a Keep entry: {missing_keep}")

# Drop the temporary normalized column
filtered = filtered.drop(columns=['datasetKey_norm'])

# Save back to the same location
filtered.to_parquet(filtered_path, index=False)
print(f"Saved updated file to: {filtered_path}")

Datasets lacking a Keep entry: 599
Saved updated file to: GBIF\datasets_GBIF_OBIS_filtered.parquet


Go through datasets with missing classification manually, save as datasets_GBIF_OBIS_manual.parquet

In [ ]:
# Load the datasets
manual_path = Path("GBIF/datasets_GBIF_OBIS_manual.parquet")
merged_path = Path("All_data/merged_all_deduplicated.parquet")

manual = pd.read_parquet(manual_path)
merged_deduped = pd.read_parquet(merged_path)

# Normalize datasetKey in both for matching
manual['datasetKey_norm'] = _norm_series(manual['datasetKey'])
merged_deduped['datasetKey_norm'] = _norm_series(merged_deduped['datasetKey'])

# Merge the Keep column from manual
merged_deduped = merged_deduped.merge(
    manual[['datasetKey_norm', 'Keep']],
    on='datasetKey_norm',
    how='left'
)

# Drop the temporary normalized column
merged_deduped = merged_deduped.drop(columns=['datasetKey_norm'])

# Save back to the same location
merged_deduped.to_parquet(merged_path, index=False)
print(f"Saved updated file to: {merged_path}")

Remove records with R or U

In [ ]:
# Load the deduplicated dataset
merged_deduped = pd.read_parquet(merged_path)

# Filter out rows where Keep column contains 'R' or 'U'
merged_filtered = merged_deduped[~merged_deduped['Keep'].isin(['R', 'U'])].copy()

print(f"Original rows: {len(merged_deduped)}")
print(f"Removed rows (Keep='R' or 'U'): {len(merged_deduped) - len(merged_filtered)}")
print(f"Remaining rows: {len(merged_filtered)}")

# Save back to the same location
merged_filtered.to_parquet(merged_path, index=False)
print(f"Saved filtered file to: {merged_path}")

In [25]:
# Debug: verify OGSL rows removed in cell 8 are exact key matches to kept rows
if 'removed_within' not in globals() or 'to_deduplicate_within_deduped' not in globals():
    print('Run cell 8 first so removed_within and to_deduplicate_within_deduped are available.')
else:
    ogsl_removed = removed_within[removed_within['sourceDatabase_norm'] == 'OGSL'].copy()
    print(f"OGSL rows removed in cell 8 run: {len(ogsl_removed)}")

    if len(ogsl_removed) == 0:
        print('No OGSL rows were removed in the current cell 8 run.')
    else:
        key_cols_verify = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']
        ogsl_kept = to_deduplicate_within_deduped[to_deduplicate_within_deduped['sourceDatabase_norm'] == 'OGSL'][key_cols_verify].drop_duplicates().copy()

        check = ogsl_removed[key_cols_verify].merge(
            ogsl_kept.assign(_has_kept_match=True),
            on=key_cols_verify,
            how='left'
        )

        missing_match = check[check['_has_kept_match'].isna()].copy()
        print(f"Removed OGSL rows with exact kept-key match: {(check['_has_kept_match'].notna()).sum()} / {len(check)}")

        if len(missing_match) > 0:
            print('\nRows that do NOT have an exact kept-key match (should be 0):')
            print(missing_match.head(20).to_string(index=False))
        else:
            print('\nAll removed OGSL rows are exact duplicates of a kept OGSL row on cell 8 key columns.')
            print('\nRemoved OGSL key rows:')
            print(check[key_cols_verify].drop_duplicates().to_string(index=False))

OGSL rows removed in cell 8 run: 3
Removed OGSL rows with exact kept-key match: 3 / 3

All removed OGSL rows are exact duplicates of a kept OGSL row on cell 8 key columns.

Removed OGSL key rows:
sourceDatabase_norm   year  startDayOfYear  decimalLongitudeStart  decimalLatitudeStart  aphiaID
               OGSL 2014.0           213.0             -77.581750             76.353367 139000.0
               OGSL 2014.0           212.0             -74.607133             76.272600 141999.0
               OGSL 2014.0           218.0             -75.722200             77.323633 141999.0
